# Sampling constrained hidden paths

This notebook demonstraed exact sampling from the constrained posterior.

- **`ffbs_torch_mvr_chmm`** draws full paths of length $T$ from $P(x \mid y, C)$.
  Every drawn path satisfies every constraint, with **no rejection**.
- **`stopped_sampling_torch_mvr_chmm`** first draws a stopping time $\tau$ from a target
  constraint's first-satisfaction-time distribution, then draws the path $x[0..\tau]$. Sampled paths have varying length. Note that the sampled path is conditioned on the
  **whole** observation sequence, including the part after $\tau$.

Two constraints appear below: `forbid_A` (*never visit `A`*) for the FFBS half, and
`reach_B` (*visit `B` at some point*) as the stopping target for the second half.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

from conin.hidden_markov_model.hmm import HiddenMarkovModel
from conin.hidden_markov_model.mvr import HomMVR
from conin.hidden_markov_model.chmm_mvr import MVR_CHMM
from conin.hidden_markov_model.sampling.ffbs_mvr import ffbs_torch_mvr_chmm
from conin.hidden_markov_model.sampling.stopped_sampling_mvr import (
    stopped_sampling_torch_mvr_chmm,
)
from conin.hidden_markov_model.other_queries.sat_time_mvr import (
    sat_time_torch_mvr_chmm,
)

In [ ]:
HIDDEN_STATES = ["A", "B", "C"]
OBSERVED_STATES = ["lo", "mid", "hi"]

hmm = HiddenMarkovModel()
hmm.load_model(
    start_probs={
        "A": 0.2765440507007986,
        "B": 0.4033576072467887,
        "C": 0.32009834205241255,
    },
    transition_probs={
        ("A", "A"): 0.3391777054270445,
        ("A", "B"): 0.049711711669595204,
        ("A", "C"): 0.6111105829033604,
        ("B", "A"): 0.48102507253852517,
        ("B", "B"): 0.05601918704283972,
        ("B", "C"): 0.4629557404186351,
        ("C", "A"): 0.43616112524444134,
        ("C", "B"): 0.1773076392327265,
        ("C", "C"): 0.38653123552283214,
    },
    emission_probs={
        ("A", "lo"): 0.19949219710155375,
        ("A", "mid"): 0.30789837305397333,
        ("A", "hi"): 0.492609429844473,
        ("B", "lo"): 0.534907622618408,
        ("B", "mid"): 0.234417585356662,
        ("B", "hi"): 0.23067479202493,
        ("C", "lo"): 0.09093879934300991,
        ("C", "mid"): 0.008996844382398088,
        ("C", "hi"): 0.9000643562745919,
    },
    initialize=True,
)

observed = ["hi", "mid", "lo", "lo", "lo", "lo", "lo"]
T = len(observed)


def reach_mvr(state, time_range=None, name=None):
    """MVR accepting once ``state`` has been visited; acceptance is absorbing."""
    mediation_states = ["not_yet", "seen"]

    return HomMVR(
        hidden_states=HIDDEN_STATES,
        mediation_states=mediation_states,
        ini={h: ("seen" if h == state else "not_yet") for h in HIDDEN_STATES},
        upd={
            (m, h): ("seen" if m == "seen" or h == state else "not_yet")
            for m in mediation_states
            for h in HIDDEN_STATES
        },
        evl={"not_yet": False, "seen": True},
        time_range=time_range,
        name=name,
    )


def forbid_mvr(state, name=None):
    """MVR rejecting any path that visits ``state``."""
    mediation_states = ["ok", "violated"]

    return HomMVR(
        hidden_states=HIDDEN_STATES,
        mediation_states=mediation_states,
        ini={h: ("violated" if h == state else "ok") for h in HIDDEN_STATES},
        upd={
            (m, h): ("violated" if m == "violated" or h == state else "ok")
            for m in mediation_states
            for h in HIDDEN_STATES
        },
        evl={"ok": True, "violated": False},
        name=name,
    )


# An empty constraint list is the unconstrained baseline every comparison below uses.
base = MVR_CHMM(hidden_markov_model=hmm, constraints=[])
forbidA = MVR_CHMM(hidden_markov_model=hmm, constraints=[forbid_mvr("A", name="forbid_A")])
reachB = MVR_CHMM(hidden_markov_model=hmm, constraints=[reach_mvr("B", name="reach_B")])

print("t   :", "  ".join(f"{t:>3}" for t in range(T)))
print("obs :", "  ".join(f"{o:>3}" for o in observed))

## FFBS

`ffbs_torch_mvr_chmm(model, observed, num_samples=..., generator=...)` returns a list
of `num_samples` paths in external state labels. Pass a seeded `torch.Generator` for
reproducibility.

In [ ]:
g = torch.Generator().manual_seed(0)

print(f"{'unconstrained':<25}forbid_A enforced")
for u, c in zip(
    ffbs_torch_mvr_chmm(base, observed, num_samples=4, generator=g),
    ffbs_torch_mvr_chmm(forbidA, observed, num_samples=4, generator=g),
):
    print("  ".join(u), "    ", "  ".join(c))

N = 20_000
U = ffbs_torch_mvr_chmm(base, observed, num_samples=N, generator=g)
C = ffbs_torch_mvr_chmm(forbidA, observed, num_samples=N, generator=g)

print(f"\npaths visiting A   unconstrained {np.mean(['A' in p for p in U]):.4f}")
print(f"                   constrained   {np.mean(['A' in p for p in C]):.4f}")

In [ ]:
def occupancy(sample):
    """P(x_t = h) estimated from a list of sampled paths."""
    return np.array(
        [[np.mean([p[t] == h for p in sample]) for t in range(T)] for h in HIDDEN_STATES]
    )


occ_u, occ_c = occupancy(U), occupancy(C)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.1), sharey=True)

for ax, h, u, c in zip(axes, HIDDEN_STATES, occ_u, occ_c):
    ax.plot(range(T), u, "o--", color="0.6", linewidth=1.6, label="unconstrained")
    ax.plot(range(T), c, "o-", color="#2E6DB4", linewidth=2, label="forbid_A")
    ax.set_title(f"$P(x_t = {h})$", fontsize=11)
    ax.set_xlabel("t")
    ax.grid(color="0.92", linewidth=0.8)
    ax.set_axisbelow(True)

axes[0].set_ylabel("posterior occupancy")
axes[0].legend(fontsize=9)
fig.suptitle(
    "A is excluded exactly, and its mass is redistributed onto B and C",
    fontsize=11, x=0.01, ha="left",
)
fig.tight_layout()
plt.show()

## Stopped sampling

A different question, and a different constraint — `reach_B`: *how did the path get to
`B` the first time?* The
stopping time $\tau$ is drawn from the target's first-satisfaction-time distribution,
so the returned paths have varying length and each ends the moment `reach_B` first
accepts.

In [ ]:
g = torch.Generator().manual_seed(0)

paths, taus = stopped_sampling_torch_mvr_chmm(
    reachB, observed, target="reach_B", num_samples=6, generator=g, return_times=True
)

for p, tau in zip(paths, taus.tolist()):
    print(f"tau = {tau}   len = {len(p)}   {p}")

In [ ]:
g = torch.Generator().manual_seed(0)
_, taus = stopped_sampling_torch_mvr_chmm(
    reachB, observed, target="reach_B", num_samples=N, generator=g, return_times=True
)
empirical = np.bincount(taus.numpy(), minlength=T) / N

# Exact, from the satisfaction-time query.
times, exact = sat_time_torch_mvr_chmm(reachB, observed, target="reach_B")

# Unconstrained baseline: first index of B among draws that reach B at all -- what
# rejection sampling would give, after discarding the ~13% that never do.
hits = [p.index("B") for p in U if "B" in p]
rejection = np.bincount(hits, minlength=T) / len(hits)

fig, ax = plt.subplots(figsize=(9, 3.8))

ax.bar(times, empirical, color="#2E6DB4", label=f"stopped sampling (N={N:,})")
ax.plot(times, exact.numpy(), "o-", color="#C4432B", linewidth=2, label="sat_time (exact)")
ax.plot(times, rejection, "s--", color="0.45", linewidth=1.6,
        label="unconstrained + rejection")

ax.set_xlabel(r"$\tau$  (first time the path visits B)")
ax.set_ylabel("probability")
ax.legend(fontsize=9)
ax.grid(axis="y", color="0.92", linewidth=0.8)
ax.set_axisbelow(True)
fig.tight_layout()
plt.show()

print(f"max |stopped sampling - exact|      = {np.abs(empirical - exact.numpy()).max():.4f}")
print(f"max |rejection baseline - exact|    = {np.abs(rejection - exact.numpy()).max():.4f}")
print(f"draws wasted by rejection           = {1 - len(hits) / N:.1%}")

## Notes

- **The prefix is conditioned on everything.** $x[0..\tau]$ is drawn given *all* of
  $y[0..T-1]$ and every constraint over the whole horizon, so an observation later
  than $\tau$ still informs it. This is deliberately not legacy
  `variable_length_sampling`, which rebuilt a shorter model and threw that away.
- **The two samplers agree with the exact answer**, which is the point of the plot
  above: stopped sampling reproduces the rejection baseline without discarding draws.
- `return_augmented` defaults to `False` here, unlike `viterbi_torch_mvr_chmm` —
  decoding mediation labels is a per-sample Python loop and usually not wanted.
- `min_length` bounds the **returned prefix**, and stopped sampling runs one forward
  pass per *distinct* drawn $\tau$, so a long horizon with a spread-out $\tau$ is the
  case to watch.